# DiffAE on CIFAR-10 (Fixed)

**Implementation of Diffusion Autoencoders on CIFAR-10**

Architecture:
- **Encoder**: Image → Semantic Latent (256-D)
- **Decoder**: (Noisy Image + Latent + Time) → Denoised Image
- **Diffusion**: DDPM/DDIM framework

Date: 2025-10-24

**Fix**: Separate EncoderBlock without time embeddings

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data Loading

In [ ]:
# CIFAR-10 dataset (32×32 RGB images)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Scale to [-1, 1]
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Visualize samples
def show_images(images, title="Images"):
    images = (images + 1) / 2  # Denormalize to [0, 1]
    grid = torchvision.utils.make_grid(images[:16], nrow=4)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).cpu())
    plt.title(title)
    plt.axis('off')
    plt.show()

sample_images, _ = next(iter(train_loader))
show_images(sample_images, "CIFAR-10 Samples")

## 3. Architecture Components

### 3.1 Basic Building Blocks

In [ ]:
def timestep_embedding(timesteps, dim, max_period=10000):
    """Create sinusoidal timestep embeddings."""
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0, end=half, dtype=torch.float32) / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class EncoderBlock(nn.Module):
    """Residual block for encoder (no time/semantic conditioning)"""
    def __init__(self, in_channels, out_channels, dropout=0.1):
        super().__init__()
        self.in_layers = nn.Sequential(
            nn.GroupNorm(32, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, padding=1)
        )
        self.out_layers = nn.Sequential(
            nn.GroupNorm(32, out_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Conv2d(out_channels, out_channels, 3, padding=1)
        )
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        h = self.in_layers(x)
        h = self.out_layers(h)
        return h + self.shortcut(x)


class ResBlock(nn.Module):
    """Residual block with time and semantic conditioning (for decoder)"""
    def __init__(self, in_channels, out_channels, time_emb_dim, cond_dim=None, dropout=0.1):
        super().__init__()
        self.use_cond = cond_dim is not None
        
        self.in_layers = nn.Sequential(
            nn.GroupNorm(32, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, 3, padding=1)
        )
        
        self.emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        
        if self.use_cond:
            self.cond_layers = nn.Sequential(
                nn.SiLU(),
                nn.Linear(cond_dim, out_channels)
            )
        
        self.out_layers = nn.Sequential(
            nn.GroupNorm(32, out_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Conv2d(out_channels, out_channels, 3, padding=1)
        )
        
        if in_channels != out_channels:
            self.shortcut = nn.Conv2d(in_channels, out_channels, 1)
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x, time_emb, cond=None):
        h = self.in_layers(x)
        h = h + self.emb_layers(time_emb)[:, :, None, None]
        if self.use_cond and cond is not None:
            scale = self.cond_layers(cond)[:, :, None, None]
            h = h * (1 + scale)
        h = self.out_layers(h)
        return h + self.shortcut(x)


class AttentionBlock(nn.Module):
    """Self-attention block"""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(32, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
    
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)
        
        q = q.view(B, self.num_heads, C // self.num_heads, H * W).transpose(2, 3)
        k = k.view(B, self.num_heads, C // self.num_heads, H * W).transpose(2, 3)
        v = v.view(B, self.num_heads, C // self.num_heads, H * W).transpose(2, 3)
        
        scale = (C // self.num_heads) ** -0.5
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * scale, dim=-1)
        h = torch.matmul(attn, v)
        
        h = h.transpose(2, 3).contiguous().view(B, C, H, W)
        h = self.proj(h)
        return x + h


class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    
    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    
    def forward(self, x):
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        return self.conv(x)

### 3.2 Semantic Encoder (Fixed)

In [ ]:
class SemanticEncoder(nn.Module):
    """Encoder: Image → Semantic Latent (no time embeddings)"""
    def __init__(self, in_channels=3, latent_dim=256, base_channels=64):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.init_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        
        # Use EncoderBlock instead of ResBlock
        self.down1 = nn.Sequential(
            EncoderBlock(base_channels, base_channels),
            EncoderBlock(base_channels, base_channels),
            Downsample(base_channels)
        )
        
        self.down2 = nn.Sequential(
            EncoderBlock(base_channels, base_channels * 2),
            EncoderBlock(base_channels * 2, base_channels * 2),
            Downsample(base_channels * 2)
        )
        
        self.down3 = nn.Sequential(
            EncoderBlock(base_channels * 2, base_channels * 4),
            EncoderBlock(base_channels * 4, base_channels * 4),
            Downsample(base_channels * 4)
        )
        
        self.down4 = nn.Sequential(
            EncoderBlock(base_channels * 4, base_channels * 4),
            EncoderBlock(base_channels * 4, base_channels * 4),
            Downsample(base_channels * 4)
        )
        
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(base_channels * 4, latent_dim)
    
    def forward(self, x):
        h = self.init_conv(x)
        h = self.down1(h)
        h = self.down2(h)
        h = self.down3(h)
        h = self.down4(h)
        h = self.pool(h).squeeze(-1).squeeze(-1)
        latent = self.proj(h)
        return latent

### 3.3 Conditional Decoder (U-Net)

In [ ]:
class ConditionalDecoder(nn.Module):
    """Decoder: (Noisy Image + Latent + Time) → Denoised Image"""
    def __init__(self, in_channels=3, out_channels=3, latent_dim=256, 
                 base_channels=64, time_emb_dim=256):
        super().__init__()
        self.time_emb_dim = time_emb_dim
        
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim * 4),
            nn.SiLU(),
            nn.Linear(time_emb_dim * 4, time_emb_dim)
        )
        
        self.init_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        
        self.down1 = nn.ModuleList([
            ResBlock(base_channels, base_channels, time_emb_dim, latent_dim),
            ResBlock(base_channels, base_channels, time_emb_dim, latent_dim),
            Downsample(base_channels)
        ])
        
        self.down2 = nn.ModuleList([
            ResBlock(base_channels, base_channels * 2, time_emb_dim, latent_dim),
            ResBlock(base_channels * 2, base_channels * 2, time_emb_dim, latent_dim),
            Downsample(base_channels * 2)
        ])
        
        self.down3 = nn.ModuleList([
            ResBlock(base_channels * 2, base_channels * 4, time_emb_dim, latent_dim),
            ResBlock(base_channels * 4, base_channels * 4, time_emb_dim, latent_dim),
            Downsample(base_channels * 4)
        ])
        
        self.middle = nn.ModuleList([
            ResBlock(base_channels * 4, base_channels * 4, time_emb_dim, latent_dim),
            AttentionBlock(base_channels * 4),
            ResBlock(base_channels * 4, base_channels * 4, time_emb_dim, latent_dim)
        ])
        
        self.up3 = nn.ModuleList([
            ResBlock(base_channels * 8, base_channels * 4, time_emb_dim, latent_dim),
            ResBlock(base_channels * 4, base_channels * 4, time_emb_dim, latent_dim),
            Upsample(base_channels * 4)
        ])
        
        self.up2 = nn.ModuleList([
            ResBlock(base_channels * 6, base_channels * 2, time_emb_dim, latent_dim),
            ResBlock(base_channels * 2, base_channels * 2, time_emb_dim, latent_dim),
            Upsample(base_channels * 2)
        ])
        
        self.up1 = nn.ModuleList([
            ResBlock(base_channels * 3, base_channels, time_emb_dim, latent_dim),
            ResBlock(base_channels, base_channels, time_emb_dim, latent_dim),
            Upsample(base_channels)
        ])
        
        self.out = nn.Sequential(
            nn.GroupNorm(32, base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, out_channels, 3, padding=1)
        )
    
    def forward(self, x, t, cond):
        t_emb = timestep_embedding(t, self.time_emb_dim)
        t_emb = self.time_mlp(t_emb)
        
        h = self.init_conv(x)  # 64ch @ 32×32
        skips = []
        
        # Down1: process @ 32×32, downsample to 16×16, save @ 16×16
        for block in self.down1[:-1]:
            h = block(h, t_emb, cond)  # 64ch @ 32×32
        h = self.down1[-1](h)  # Downsample to 16×16
        skips.append(h)  # Save 64ch @ 16×16
        
        # Down2: process @ 16×16, downsample to 8×8, save @ 8×8
        for block in self.down2[:-1]:
            h = block(h, t_emb, cond)  # 128ch @ 16×16
        h = self.down2[-1](h)  # Downsample to 8×8
        skips.append(h)  # Save 128ch @ 8×8
        
        # Down3: process @ 8×8, downsample to 4×4, save @ 4×4
        for block in self.down3[:-1]:
            h = block(h, t_emb, cond)  # 256ch @ 8×8
        h = self.down3[-1](h)  # Downsample to 4×4
        skips.append(h)  # Save 256ch @ 4×4
        
        # Middle: 256ch @ 4×4
        for block in self.middle:
            if isinstance(block, ResBlock):
                h = block(h, t_emb, cond)
            else:
                h = block(h)
        
        # Up3: concat @ 4×4, process, upsample to 8×8
        h = torch.cat([h, skips.pop()], dim=1)  # (256+256=512ch) @ 4×4
        for block in self.up3[:-1]:
            h = block(h, t_emb, cond)  # 512→256→256ch @ 4×4
        h = self.up3[-1](h)  # Upsample to 8×8
        
        # Up2: concat @ 8×8, process, upsample to 16×16
        h = torch.cat([h, skips.pop()], dim=1)  # (256+128=384ch) @ 8×8
        for block in self.up2[:-1]:
            h = block(h, t_emb, cond)  # 384→128→128ch @ 8×8
        h = self.up2[-1](h)  # Upsample to 16×16
        
        # Up1: concat @ 16×16, process, upsample to 32×32
        h = torch.cat([h, skips.pop()], dim=1)  # (128+64=192ch) @ 16×16
        for block in self.up1[:-1]:
            h = block(h, t_emb, cond)  # 192→64→64ch @ 16×16
        h = self.up1[-1](h)  # Upsample to 32×32
        
        return self.out(h)

### 3.4 Complete DiffAE Model

In [ ]:
class DiffusionAutoencoder(nn.Module):
    def __init__(self, latent_dim=256, base_channels=64):
        super().__init__()
        self.encoder = SemanticEncoder(3, latent_dim, base_channels)
        self.decoder = ConditionalDecoder(3, 3, latent_dim, base_channels)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, x_noisy, t, latent):
        return self.decoder(x_noisy, t, latent)
    
    def forward(self, x_noisy, t, x_clean):
        latent = self.encode(x_clean)
        pred = self.decode(x_noisy, t, latent)
        return pred


# Test model
model = DiffusionAutoencoder(latent_dim=256, base_channels=64).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

# Test forward pass
x_test = torch.randn(4, 3, 32, 32).to(device)
t_test = torch.randint(0, 1000, (4,)).to(device)
latent_test = model.encode(x_test)
pred_test = model.decode(x_test, t_test, latent_test)
print(f"✅ Input shape: {x_test.shape}")
print(f"✅ Latent shape: {latent_test.shape}")
print(f"✅ Output shape: {pred_test.shape}")
print("\n🎉 Model created successfully!")